# Diet & Nutrition Chatbot
**Subject:** Natural Language Understanding (NLU)
**Model:** llama-3.3-70b-versatile via Groq API

Tell the chatbot what you ate today and how much. It will calculate your total calories, fat, protein, and carbs. You can also ask for diet guidance like how to lose fat or gain weight.

## Cell 1 - Install Dependencies

In [1]:
# Install required packages
# groq      : Groq API SDK to call the LLM
# ipywidgets: for building the interactive chat UI inside Colab

!pip install groq ipywidgets --quiet

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.5 MB/s eta 0:00:00
Installation complete.


## Cell 2 - Import Libraries

In [2]:
import json
from getpass import getpass
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from groq import Groq

print("All libraries imported.")

All libraries imported.


## Cell 3 - Enter Groq API Key

Get a free API key at [console.groq.com](https://console.groq.com) (no credit card needed).
1. Sign up and go to API Keys
2. Click Create API Key
3. Copy the key (starts with gsk_) and paste below when prompted

In [3]:
# Ask user to enter their Groq API key securely
# getpass hides the input so the key is not visible in the notebook

GROQ_API_KEY = getpass("Paste your Groq API Key: ")

if GROQ_API_KEY.strip().startswith("gsk_"):
    print("API key looks valid.")
elif GROQ_API_KEY.strip():
    print("Key entered but does not start with 'gsk_' - double check it.")
else:
    print("No key entered. Re-run this cell and paste your key.")

Paste your Groq API Key: ··········
API key looks valid.


## Cell 4 - Chatbot Logic

In [4]:
# This is the system prompt that tells the LLM what role to play
# It defines the chatbot behavior - nutrition expert, what to track, how to respond
SYSTEM_PROMPT = """You are a helpful nutrition and diet assistant.

Your job is to help users track what they eat and understand their nutrition.

When a user tells you what they ate, you must:
1. Estimate the calories, fat (g), protein (g), and carbohydrates (g) for each food item
2. Show a simple breakdown for each item
3. Show the total at the end

Use realistic nutritional estimates based on standard Indian and international food data.
Be accurate - do not guess wildly. If the user gives a quantity, use it. If not, assume a standard serving size.

When a user asks for diet advice (like how to lose fat or gain weight), give practical, realistic guidance:
- Suggest a daily calorie target
- Recommend what kinds of foods to eat more or less of
- Suggest a simple daily routine (meals, water, activity)
- Keep it simple and actionable, not overwhelming

Rules:
- Be friendly and conversational, not robotic
- Do not use excessive formatting like headers everywhere
- Do not say things like "As an AI language model..."
- Keep responses clear and to the point
- If something is unclear, ask the user to clarify
"""

# Stores the full conversation history so the LLM has context across messages
conversation_history = []

def chat(user_message, api_key):
    """
    Sends the user message to Groq API and returns the assistant reply.
    Keeps full conversation history so the LLM remembers what was said earlier.
    """
    # Add the new user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    client = Groq(api_key=api_key.strip())

    # Send the full conversation history along with the system prompt
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history,
        temperature=0.6,
        max_tokens=1024
    )

    reply = response.choices[0].message.content

    # Add assistant reply to history so it is included in the next API call
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

print("Chatbot functions ready.")

Chatbot functions ready.


## Cell 5 - Chat Interface

Run this cell to open the chatbot. Type what you ate and press Enter or click Send.

Examples of what you can say:
- "I ate 2 rotis, 1 bowl of dal, and a cup of rice for lunch"
- "Had 2 boiled eggs and a banana for breakfast"
- "How should I eat to lose fat?"
- "What should my diet look like to gain weight?"

In [5]:
# Check that the API key was set before launching the UI
if "GROQ_API_KEY" not in dir() or not GROQ_API_KEY.strip():
    print("API key not found. Please run Cell 3 first.")
else:
    # Reset conversation when UI is launched so each session starts fresh
    conversation_history.clear()

    # -- Build the chat interface using ipywidgets --------------------------

    # Chat history display area (read-only, shows the conversation)
    chat_output = widgets.Output(
        layout=widgets.Layout(
            height="420px",
            overflow_y="auto",
            border="1px solid #d0d0d0",
            padding="12px",
            background_color="#fafafa"
        )
    )

    # Text input box where user types their message
    user_input = widgets.Text(
        placeholder="Type what you ate today, or ask a diet question...",
        layout=widgets.Layout(width="78%", height="38px")
    )

    # Send button
    send_button = widgets.Button(
        description="Send",
        button_style="primary",
        layout=widgets.Layout(width="18%", height="38px")
    )

    # Clear button to reset the conversation
    clear_button = widgets.Button(
        description="Clear Chat",
        button_style="warning",
        layout=widgets.Layout(width="12%", height="32px")
    )

    # Arrange input and send button side by side
    input_row = widgets.HBox(
        [user_input, send_button],
        layout=widgets.Layout(gap="8px", margin="8px 0 4px 0")
    )

    # Helper function to add a message bubble to the chat display
    def add_message(role, text):
        """Renders a chat bubble in the output area based on who sent it."""
        if role == "user":
            bubble_style = (
                "background:#1a73e8; color:white; padding:10px 14px; "
                "border-radius:16px 16px 4px 16px; margin:6px 0 6px 20%; "
                "display:inline-block; max-width:75%; word-wrap:break-word; "
                "font-family:Arial,sans-serif; font-size:14px; line-height:1.5;"
            )
            wrapper_style = "text-align:right; margin-bottom:4px;"
            label = ""
        else:
            bubble_style = (
                "background:#f0f0f0; color:#1a1a1a; padding:10px 14px; "
                "border-radius:16px 16px 16px 4px; margin:6px 20% 6px 0; "
                "display:inline-block; max-width:75%; word-wrap:break-word; "
                "font-family:Arial,sans-serif; font-size:14px; line-height:1.5;"
            )
            wrapper_style = "text-align:left; margin-bottom:4px;"
            label = ""

        # Convert newlines to <br> so line breaks show in HTML
        formatted_text = text.replace("\n", "<br>")

        html = f"""
        <div style="{wrapper_style}">
            <div style="{bubble_style}">{formatted_text}</div>
        </div>
        """
        with chat_output:
            display(HTML(html))

    # Function that runs when the user sends a message
    def on_send(event=None):
        message = user_input.value.strip()
        if not message:
            return

        # Show the user's message in the chat
        add_message("user", message)
        user_input.value = ""

        # Show a loading indicator while waiting for the API response
        with chat_output:
            display(HTML(
                "<div style='color:#888; font-size:13px; font-family:Arial; "
                "margin:4px 0 4px 0;'>Thinking...</div>"
            ))

        # Call the LLM and get the reply
        try:
            reply = chat(message, GROQ_API_KEY)
        except Exception as e:
            reply = f"Something went wrong: {str(e)}"

        # Remove the "Thinking..." indicator and show the actual reply
        with chat_output:
            clear_output(wait=True)

        # Re-render the full conversation history (since we cleared the output)
        for msg in conversation_history:
            add_message(msg["role"], msg["content"])

    # Function that runs when Clear Chat is clicked
    def on_clear(event=None):
        conversation_history.clear()
        with chat_output:
            clear_output()
        # Show welcome message again
        add_message("assistant",
            "Chat cleared. Tell me what you ate today and I will calculate your nutrition.\n"
            "For example: 'I had 2 rotis, 1 bowl of dal, and a cup of rice for lunch.'"
        )

    # Connect buttons and Enter key to their handler functions
    send_button.on_click(on_send)
    clear_button.on_click(on_clear)
    user_input.on_submit(on_send)

    # -- Build final layout -------------------------------------------------
    header = widgets.HTML(
        "<div style='font-family:Arial,sans-serif; font-size:18px; font-weight:600; "
        "color:#1a1a1a; margin-bottom:4px;'>Diet & Nutrition Chatbot</div>"
        "<div style='font-family:Arial,sans-serif; font-size:13px; color:#666; "
        "margin-bottom:10px;'>Tell me what you ate today. I will track your calories, fat, protein, and carbs.</div>"
    )

    controls_row = widgets.HBox(
        [clear_button],
        layout=widgets.Layout(margin="0 0 4px 0")
    )

    ui = widgets.VBox(
        [header, chat_output, controls_row, input_row],
        layout=widgets.Layout(
            width="700px",
            padding="16px",
            border="1px solid #e0e0e0",
            border_radius="12px",
            background_color="white"
        )
    )

    display(ui)

    # Show the welcome message when the chat first opens
    add_message("assistant",
        "Hello! I am your diet and nutrition assistant.\n\n"
        "Tell me what you ate today and I will calculate your total calories, fat, protein, and carbs.\n\n"
        "You can also ask me things like:\n"
        "- How should I eat to lose fat?\n"
        "- What is a good diet to gain weight?\n"
        "- How many calories should I eat per day?"
    )